# 3. Spark SQL, joins and window functions

The code of this module's lessons in [Learn Apache Spark by Doing](https://dataenglab.com/courses/learn-apache-spark-by-doing/), in lesson order, on Apache Spark 4.2.0 in local mode: one machine, no cluster.

Colab runs this notebook in your own Google account: sign in with Google to run the cells, and use *File → Save a copy in Drive* to keep your changes. Nothing runs on DataEngLab and nothing is installed on your computer.

Run the cells from top to bottom. Under a cell, the result the lesson shows is there to compare with yours.

Not available in Colab: the Spark UI (its port is not reachable from a notebook). Where a lesson sends you to it, run that part on your machine; here, `df.explain()` prints the plan.

In [ ]:
# Setup: Java 17+, PySpark 4.2.0 and the webshop data. About a minute the first time.
import os, re, subprocess, sys, urllib.request, zipfile

PYSPARK_VERSION = "4.2.0"
DATA_URL = 'https://github.com/dataeng-lab/learn-apache-spark-by-doing/raw/v2/datasets.zip'

if "google.colab" in sys.modules:
    version = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    major = re.search(r'version "(?:1\.)?(\d+)', version)
    if not major or int(major.group(1)) < 17:       # PySpark 4 does not start on Java 8 or 11
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                       shell=True, check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"], check=True)

if not os.path.isdir("datasets"):
    if DATA_URL is None:
        raise RuntimeError("datasets/ is missing: upload the course's datasets folder in the Files panel")
    urllib.request.urlretrieve(DATA_URL, "datasets.zip")
    zipfile.ZipFile("datasets.zip").extractall(".")

print("datasets:", sorted(os.listdir("datasets")))

**Starting point.** The DataFrames and names the earlier modules built. Not part of the lesson text.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("webshop")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

from pyspark.sql.types import (
    DecimalType, IntegerType, StringType, StructField, StructType, TimestampType,
)

ORDERS_SCHEMA = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=True),
    StructField("order_ts", TimestampType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("amount", DecimalType(10, 2), nullable=True),
    StructField("currency", StringType(), nullable=True),
    StructField("updated_at", TimestampType(), nullable=True),
])
orders_schema = ORDERS_SCHEMA   # some lessons use this name

orders = spark.read.csv("datasets/orders.csv", header=True, schema=ORDERS_SCHEMA)

customers = spark.read.csv("datasets/customers.csv", header=True, inferSchema=True)
order_items = spark.read.csv("datasets/order_items.csv", header=True, inferSchema=True)

## Spark SQL and temporary views

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/spark-sql-and-temp-views/)

In [ ]:
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

The lesson saves this as `high_value_by_status.py`.

In [ ]:
orders.createOrReplaceTempView("orders")

spark.sql("""
    SELECT status, count(*) AS orders
    FROM orders
    WHERE amount > 1000
    GROUP BY status
    ORDER BY orders DESC
""").show(truncate=False)

(orders
 .filter(F.col("amount") > 1000)
 .groupBy("status")
 .agg(F.count("*").alias("orders"))
 .orderBy(F.col("orders").desc())
 .show(truncate=False))

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
Spark SQL:
+---------+------+
|status   |orders|
+---------+------+
|created  |853   |
|delivered|14    |
|paid     |12    |
|cancelled|10    |
|shipped  |9     |
+---------+------+

DataFrame API:
+---------+------+
|status   |orders|
+---------+------+
|created  |853   |
|delivered|14    |
|paid     |12    |
|cancelled|10    |
|shipped  |9     |
+---------+------+
```

## Built-in functions before UDFs

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/built-in-functions-before-udfs/)

The lesson saves this as `bucket_orders.py`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

builtin = orders.withColumn(
    "bucket",
    F.when(F.col("amount") >= 500, "large").otherwise("small"),
)

@F.udf(returnType=StringType())
def bucket_udf(amount):
    if amount is None:
        return None
    return "large" if amount >= 500 else "small"

udf_version = orders.withColumn("bucket", bucket_udf(F.col("amount")))

## Joining orders, customers and items

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/joining-the-webshop-datasets/)

The lesson saves this as `join_customers.py`.

In [ ]:
# key column appears once in the result
by_name = orders.join(customers, on="customer_id", how="inner")

# both customer_id columns survive; select("customer_id") is now ambiguous
by_expr = orders.join(
    customers,
    orders.customer_id == customers.customer_id,
    how="inner",
)

The lesson saves this as `order_detail.py`.

In [ ]:
detail = (
    orders
    .join(customers, on="customer_id", how="left")
    .join(order_items, on="order_id", how="left")
)

## Fan-out and NULLs: the two joins that lie

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/join-fan-out-and-nulls/)

The lesson saves this as `revenue_after_join.py`.

In [ ]:
joined = orders.join(order_items, on="order_id", how="inner")

print("orders rows     :", orders.count())
print("order_items rows:", order_items.count())
print("after inner join:", joined.count())

joined.agg(F.sum("amount").alias("revenue")).show()
orders.agg(F.sum("amount").alias("revenue")).show()

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
orders rows     : 4235
order_items rows: 11887
after inner join: 12567  <- more rows than orders: this is fan-out

summing amount after the join double-counts it:
  sum(amount) after join : 7893509.75
  sum(amount) on orders  : 2659175.00
```

The lesson saves this as `basket_per_order.py`.

In [ ]:
items_per_order = (
    order_items
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),
        F.sum("line_total").alias("items_total"),
    )
)

per_order = orders.join(items_per_order, on="order_id", how="left")

## Window functions: the latest state per order

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/window-functions-latest-order-state/)

The lesson saves this as `latest_state.py`.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

latest = (
    orders
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)